# Part 2 — NumPy & Pandas Data Wrangling 

# NumPy arrays: video metrics

Load the `views`, `likes`, `dislikes`, and `comment_count` columns from the GB, IN, and US CSV files, in that order. Missing or invalid metric values are replaced with `0` before converting to NumPy integer arrays. Each one-dimensional array uses the same row order, so an index refers to the same record across all four arrays.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

country_codes = ("GB", "IN", "US")
expected_files = tuple(f"{country}videos.csv" for country in country_codes)
possible_data_dirs = (
    Path.cwd(),
    *(parent / "part2_numpy_pandas" for parent in (Path.cwd(), *Path.cwd().parents)),
)
DATA_DIR = next(
    (
        directory
        for directory in possible_data_dirs
        if all((directory / filename).is_file() for filename in expected_files)
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find GBvideos.csv, INvideos.csv, and USvideos.csv "
        "in the current directory or a part2_numpy_pandas folder."
    )

metric_columns = ("views", "likes", "dislikes", "comment_count")
country_frames = []
for filename in expected_files:
    country_frame = pd.read_csv(DATA_DIR / filename, usecols=metric_columns)
    for column in metric_columns:
        country_frame[column] = (
            pd.to_numeric(country_frame[column], errors="coerce")
            .fillna(0)
            .astype(np.int64)
        )
    country_frames.append(country_frame)

video_metrics = pd.concat(country_frames, ignore_index=True)
views = video_metrics["views"].to_numpy(dtype=np.int64)
likes = video_metrics["likes"].to_numpy(dtype=np.int64)
dislikes = video_metrics["dislikes"].to_numpy(dtype=np.int64)
comment_count = video_metrics["comment_count"].to_numpy(dtype=np.int64)
metric_array = video_metrics.to_numpy(dtype=np.int64)

for name, array in (
    ("views", views),
    ("likes", likes),
    ("dislikes", dislikes),
    ("comment_count", comment_count),
):
    print(f"{name}: shape={array.shape}, dtype={array.dtype}")
print(f"Combined records: {len(video_metrics):,}")

views: shape=(46230,), dtype=int64
likes: shape=(46230,), dtype=int64
dislikes: shape=(46230,), dtype=int64
comment_count: shape=(46230,), dtype=int64
Combined records: 46,230


## Indexing and slicing

`np.sort(views)[-10:][::-1]` selects and orders the ten largest view counts. `metric_array[::100]` selects every hundredth row from the four-column array; the displayed rows include their original row indices. The final example uses row and column slices to select five consecutive records and all four metrics.

In [2]:
largest_10_views = np.sort(views)[-10:][::-1]
print("10 largest view counts:")
print(largest_10_views)
every_100th_record = metric_array[::100]

every_100th_indices = np.arange(0, len(metric_array), 100)
indexed_sample = np.column_stack((every_100th_indices, every_100th_record))
print(f"\nEvery 100th record (row index, views, likes, dislikes, comment_count); shape={every_100th_record.shape}:")
print(indexed_sample.tolist())

metric_subarray = metric_array[10:15, :]
print("\nChosen sub-array: rows 10 through 14, all four metrics:")
print(metric_subarray)

10 largest view counts:
[360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]

Every 100th record (row index, views, likes, dislikes, comment_count); shape=(463, 4):
[[0, 84344, 2155, 163, 475], [100, 656734, 31256, 3627, 2876], [200, 109438, 2796, 212, 617], [300, 289683, 12026, 324, 317], [400, 124665, 3186, 242, 703], [500, 67649, 3445, 189, 0], [600, 156685, 4004, 304, 883], [700, 79924, 1416, 120, 238], [800, 195161, 4987, 379, 1100], [900, 9083642, 421644, 10192, 7843], [1000, 207338, 12905, 2398, 811], [1100, 378844, 18415, 820, 1634], [1200, 259556, 7098, 397, 698], [1300, 193359, 5144, 177, 359], [1400, 9714, 332, 13, 36], [1500, 216727, 9105, 949, 196], [1600, 33827, 1074, 34, 131], [1700, 429799, 10908, 490, 807], [1800, 340953, 11365, 1188, 2291], [1900, 90008, 2153, 84, 178], [2000, 141471, 4364, 259, 485], [2100, 84232, 2576, 64, 547], [2200, 421423, 11715, 1038, 1139], [2300, 49866, 1011, 90, 208], [2400, 186414, 4899, 330

## Engagement score without loops

The like ratio is likes divided by total likes plus dislikes; the comment rate is comments per view. Both use `np.divide` with a zero-filled output and a `where` mask, so records with zero denominators receive `0` rather than a division error. Min-max scaling puts views, likes, and comments on a common 0–1 scale; a constant-valued metric maps to zeros. The engagement score weights normalized views at 20%, normalized likes at 50%, and normalized comments at 30%. Likes and comments receive most of the weight because they represent active responses, while views provide a smaller reach component. `np.log1p` compresses the highly skewed view-count distribution, making differences among low- and mid-view records easier to see on a plot. The final boolean mask uses the 90th-percentile score cutoff; ties at the cutoff can flag slightly more than 10% of records.

In [3]:
like_ratio = np.divide(
    likes,
    likes + dislikes,
    out=np.zeros(likes.shape, dtype=np.float64),
    where=(likes + dislikes) != 0,
)
comment_rate = np.divide(
    comment_count,
    views,
    out=np.zeros(views.shape, dtype=np.float64),
    where=views != 0,
)

def min_max_normalize(values):
    value_range = values.max() - values.min()
    return np.divide(
        values - values.min(),
        value_range,
        out=np.zeros(values.shape, dtype=np.float64),
        where=value_range != 0,
    )

views_normalized = min_max_normalize(views)
likes_normalized = min_max_normalize(likes)
comments_normalized = min_max_normalize(comment_count)

engagement_score = (
    0.20 * views_normalized
    + 0.50 * likes_normalized
    + 0.30 * comments_normalized
)
log_views = np.log1p(views)

top_10_percent_cutoff = np.percentile(engagement_score, 90)
top_10_percent_mask = engagement_score >= top_10_percent_cutoff
top_record_indices = np.flatnonzero(top_10_percent_mask)

print("Like ratio (first 5):", like_ratio[:5])
print("Comment rate (first 5):", comment_rate[:5])
print("Log-scaled views (first 5):", log_views[:5])
print("Engagement score (first 5):", engagement_score[:5])
print(
    f"Top-decile score cutoff: {top_10_percent_cutoff:.6f}; "
    f"flagged records: {len(top_record_indices):,} / {len(engagement_score):,}"
)
print("Top-record indices (first 10):", top_record_indices[:10])

Like ratio (first 5): [0.92968076 0.94252874 0.92654028 0.89732143 0.9503866 ]
Comment rate (first 5): [0.0056317  0.00739474 0.00408733 0.00834079 0.0010094 ]
Log-scaled views (first 5): [11.34267081 12.34772411 12.73332166 11.98851969 10.99275633]
Engagement score (first 5): [2.68366392e-04 8.82163586e-04 1.02632919e-03 6.82469098e-04
 8.22683191e-05]
Top-decile score cutoff: 0.005097; flagged records: 4,623 / 46,230
Top-record indices (first 10): [302 369 399 405 479 533 534 545 566 589]
